In [22]:
import pandas as pd
import numpy as np

sales    = pd.read_csv('sales_weekly.csv')
weather  = pd.read_csv('weather_weekly.csv')
holidays = pd.read_csv('holiday_weekly.csv')

counts = sales.groupby('product_id').size()
valid = counts[counts >= 52].index
sales = sales[sales['product_id'].isin(valid)].copy()

sales['week_start'] = pd.to_datetime(sales['week_start'])

sales['year'] = sales['week_start'].dt.isocalendar().year.astype(int)
sales['week'] = sales['week_start'].dt.isocalendar().week.astype(int)

print(f"Sales:    {sales.shape}")
print(f"Weather:  {weather.shape}")
print(f"Holidays: {holidays.shape}")

Sales:    (13579, 5)
Weather:  (209, 9)
Holidays: (209, 11)


In [23]:
df = sales.merge(weather,  on=['year', 'week'], how='left')
df = df.merge(holidays, on=['year', 'week'], how='left')

print(df.shape)
print(df.isnull().sum())

(13579, 21)
product_id                0
week                      0
sku_sold                  0
week_start                0
year                      0
temp_mean              3099
temp_min               3099
temp_max               3099
precip_sum             3099
sunshine_sum           3099
temp_anomaly           3099
heavy_rain             3099
has_holiday              53
min_days_to_holiday      53
hol_ascension            53
hol_christmas            53
hol_easter               53
hol_kings_day            53
hol_liberation_day       53
hol_new_year             53
hol_pentecost            53
dtype: int64


In [24]:
weather_cols = ['temp_mean', 'temp_min', 'temp_max',
                'precip_sum', 'sunshine_sum', 'temp_anomaly', 'heavy_rain']

for col in weather_cols:
    week_avg = df.groupby('week')[col].transform('mean')
    df[col] = df[col].fillna(week_avg)

print(df[weather_cols].isnull().sum())

temp_mean       0
temp_min        0
temp_max        0
precip_sum      0
sunshine_sum    0
temp_anomaly    0
heavy_rain      0
dtype: int64


In [25]:
holiday_cols = ['has_holiday', 'min_days_to_holiday', 'hol_ascension',
                'hol_christmas', 'hol_easter', 'hol_kings_day',
                'hol_liberation_day', 'hol_new_year', 'hol_pentecost']
df[holiday_cols] = df[holiday_cols].fillna(0)

In [26]:
df['month']          = df['week_start'].dt.month
df['week_of_year']   = df['week_start'].dt.isocalendar().week.astype(int)
df['is_month_start'] = (df['week_start'].dt.day <= 7).astype(int)
df['is_month_end']   = (df['week_start'].dt.day >= 24).astype(int)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['week_sin']  = np.sin(2 * np.pi * df['week_of_year'] / 52)
df['week_cos']  = np.cos(2 * np.pi * df['week_of_year'] / 52)

print(df[['month', 'week_of_year', 'month_sin', 'month_cos']].head(5))

   month  week_of_year  month_sin     month_cos
0      2             9   0.866025  5.000000e-01
1      3            10   1.000000  6.123234e-17
2      3            11   1.000000  6.123234e-17
3      3            12   1.000000  6.123234e-17
4      3            13   1.000000  6.123234e-17


In [27]:
df = df.sort_values(['product_id', 'week_start']).reset_index(drop=True)

for lag in [1, 2, 4, 8]:
    df[f'lag_{lag}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(lag)
    )

print(df[['product_id', 'week_start', 'sku_sold', 'lag_1', 'lag_2', 'lag_4', 'lag_8']].head(10))

   product_id week_start  sku_sold  lag_1  lag_2  lag_4  lag_8
0           2 2018-02-26       344    NaN    NaN    NaN    NaN
1           2 2018-03-05       369  344.0    NaN    NaN    NaN
2           2 2018-03-12       499  369.0  344.0    NaN    NaN
3           2 2018-03-19       228  499.0  369.0    NaN    NaN
4           2 2018-03-26       217  228.0  499.0  344.0    NaN
5           2 2018-04-02       285  217.0  228.0  369.0    NaN
6           2 2018-04-09       798  285.0  217.0  499.0    NaN
7           2 2018-04-16       171  798.0  285.0  228.0    NaN
8           2 2018-04-23       329  171.0  798.0  217.0  344.0
9           2 2018-04-30       173  329.0  171.0  285.0  369.0


In [28]:
for window in [4, 8]:
    df[f'rolling_mean_{window}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f'rolling_std_{window}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )

df[['rolling_std_4', 'rolling_std_8']] = df[['rolling_std_4', 'rolling_std_8']].fillna(0)

print(df[['product_id', 'week_start', 'rolling_mean_4', 'rolling_std_4']].head(10))

   product_id week_start  rolling_mean_4  rolling_std_4
0           2 2018-02-26             NaN       0.000000
1           2 2018-03-05          344.00       0.000000
2           2 2018-03-12          356.50      17.677670
3           2 2018-03-19          404.00      83.216585
4           2 2018-03-26          360.00     111.178535
5           2 2018-04-02          328.25     133.220056
6           2 2018-04-09          307.25     131.261507
7           2 2018-04-16          382.00     278.930099
8           2 2018-04-23          367.75     290.630780
9           2 2018-04-30          395.75     276.308252


In [29]:
before = len(df)
df = df.dropna(subset=['lag_1', 'lag_2', 'lag_4', 'lag_8'])
after = len(df)

print(f"Dropped {before - after} rows due to lag NaN")
print(f"Final shape: {df.shape}")
print(f"Products: {df['product_id'].nunique()}")

Dropped 912 rows due to lag NaN
Final shape: (12667, 37)
Products: 114


In [30]:
feature_cols = [

    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_mean_4', 'rolling_mean_8',
    'rolling_std_4', 'rolling_std_8',

    'month', 'week_of_year',
    'is_month_start', 'is_month_end',
    'month_sin', 'month_cos',
    'week_sin', 'week_cos',

    'temp_mean', 'temp_min', 'temp_max',
    'precip_sum', 'sunshine_sum', 'temp_anomaly', 'heavy_rain',

    'has_holiday', 'min_days_to_holiday',
    'hol_christmas', 'hol_easter', 'hol_kings_day',
    'hol_new_year', 'hol_pentecost', 'hol_ascension',
]

target_col = 'sku_sold'

print(f"Features: {len(feature_cols)}")
print(df[feature_cols].isnull().sum())

Features: 31
lag_1                  0
lag_2                  0
lag_4                  0
lag_8                  0
rolling_mean_4         0
rolling_mean_8         0
rolling_std_4          0
rolling_std_8          0
month                  0
week_of_year           0
is_month_start         0
is_month_end           0
month_sin              0
month_cos              0
week_sin               0
week_cos               0
temp_mean              0
temp_min               0
temp_max               0
precip_sum             0
sunshine_sum           0
temp_anomaly           0
heavy_rain             0
has_holiday            0
min_days_to_holiday    0
hol_christmas          0
hol_easter             0
hol_kings_day          0
hol_new_year           0
hol_pentecost          0
hol_ascension          0
dtype: int64


In [31]:
df.to_csv('features_final.csv', index=False)
print(f"Saved: features_final.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved: features_final.csv
Shape: (12667, 37)
Columns: ['product_id', 'week', 'sku_sold', 'week_start', 'year', 'temp_mean', 'temp_min', 'temp_max', 'precip_sum', 'sunshine_sum', 'temp_anomaly', 'heavy_rain', 'has_holiday', 'min_days_to_holiday', 'hol_ascension', 'hol_christmas', 'hol_easter', 'hol_kings_day', 'hol_liberation_day', 'hol_new_year', 'hol_pentecost', 'month', 'week_of_year', 'is_month_start', 'is_month_end', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'rolling_mean_4', 'rolling_std_4', 'rolling_mean_8', 'rolling_std_8']
